In [ ]:
# =====================================
# 📥 EEG Data Loading (Relative Paths)
# =====================================

import os
import glob
import pandas as pd

def load_eeg_data(data_root):
    """
    Load EEG data from anonymized participant folders.

    Args:
        data_root (str): Root directory containing participant folders (e.g., anonymized_data/p1, p2, ...)

    Returns:
        all_data (pd.DataFrame): Combined dataframe containing all EEG data
                                with patient_id and recording_id columns.
        data_dict (dict): Dictionary mapping recording_id -> dataframe.
    """

    channels = [
        "EEG.AF3", "EEG.F7", "EEG.F3", "EEG.FC5", "EEG.T7", "EEG.P7",
        "EEG.O1", "EEG.O2", "EEG.P8", "EEG.T8", "EEG.FC6", "EEG.F4",
        "EEG.F8", "EEG.AF4", "MarkerValueInt"
    ]

    all_data = []
    data_dict = {}

    # Find all CSV files recursively
    file_paths = glob.glob(
        os.path.join(data_root, "**", "*.csv"),
        recursive=True
    )

    for path in file_paths:
        # Extract identifiers
        recording_id = os.path.splitext(os.path.basename(path))[0]
        patient_id = os.path.basename(os.path.dirname(path))  # e.g., p3, p11

        # Load CSV
        df = pd.read_csv(path, skiprows=1, usecols=channels)

        # Add identifiers
        df["patient_id"] = patient_id
        df["recording_id"] = recording_id

        # Store
        data_dict[recording_id] = df
        all_data.append(df)

    all_data = pd.concat(all_data, ignore_index=True)

    print(f"✅ Loaded {len(file_paths)} recordings")
    print(f"🔹 Combined dataset shape: {all_data.shape}")
    print("🔹 Columns:", list(all_data.columns))

    return all_data, data_dict


# =====================================
# 📂 Dataset Root (Relative Path)
# =====================================

DATA_ROOT = "anonymized_data"

all_data, data_dict = load_eeg_data(DATA_ROOT)

: 

In [2]:
# =====================================
# 🏷️ EEG Label Mapping Function
# =====================================

def replace_marker_with_label(all_data, data_dict):
    """
    Replace numeric MarkerValueInt values with descriptive MI labels.
    Unlabeled samples are preserved.
    """

    label_map = {
        82: "Foot Dorsiflexion",
        25: "Shoulder Shrug",
        24: "Eyebrow Raise",
        23: "Left Hand Grasp",
        5:  "Knee Extension",
        9:  "Right Hand Grasp",
        7:  "Lip Purse",
        2:  "Jaw Clench",
        1:  "Fail"
    }

    updated_all_data = all_data.copy()
    updated_all_data["Label"] = updated_all_data["MarkerValueInt"].map(label_map)
    updated_all_data.drop(columns=["MarkerValueInt"], inplace=True)

    updated_data_dict = {}
    for name, df in data_dict.items():
        df_copy = df.copy()
        df_copy["Label"] = df_copy["MarkerValueInt"].map(label_map)
        df_copy.drop(columns=["MarkerValueInt"], inplace=True)
        updated_data_dict[name] = df_copy

    print("✅ Replaced numeric markers with descriptive labels (unlabeled samples preserved).")
    return updated_all_data, updated_data_dict

all_data, data_dict = replace_marker_with_label(all_data, data_dict)

✅ Replaced numeric markers with descriptive labels (unlabeled samples preserved).


In [3]:
# =====================================
# 🧩 EEG Window Segmentation Function (Patient-Aware)
# =====================================

import numpy as np
import pandas as pd

def create_timeframes(df, sampling_rate=256, pre_seconds=2, post_seconds=3):
    """
    Split continuous EEG data into windows around labeled events.
    Preserves patient identity per window.
    """

    channels = [col for col in df.columns if col.startswith("EEG.")]
    pre_samples = int(pre_seconds * sampling_rate)
    post_samples = int(post_seconds * sampling_rate)

    X_windows, y_labels, patient_ids = [], [], []

    labeled_indices = df.index[df["Label"].notna()]

    for idx in labeled_indices:
        label = df.at[idx, "Label"]

        if label == "Fail":
            if X_windows:
                X_windows.pop()
                y_labels.pop()
                patient_ids.pop()
            continue

        start_idx = idx - pre_samples
        end_idx = idx + post_samples

        if start_idx < 0 or end_idx >= len(df):
            continue

        segment = df.iloc[start_idx:end_idx][channels].values
        patient_id = df.at[idx, "patient_id"]

        X_windows.append(segment)
        y_labels.append(label)
        patient_ids.append(patient_id)

    return X_windows, y_labels, patient_ids


# =====================================
# 🔁 Create windows for all recordings
# =====================================

X_all, y_all, patient_ids = [], [], []

for _, df_rec in data_dict.items():
    X_s, y_s, pid_s = create_timeframes(df_rec, sampling_rate=256)
    X_all.extend(X_s)
    y_all.extend(y_s)
    patient_ids.extend(pid_s)

print(f"✅ Total windows: {len(X_all)}")
print(f"✅ Unique patients: {len(set(patient_ids))}")

✅ Total windows: 1716
✅ Unique patients: 11


In [4]:
# =====================================
# 🔀 Flexible Train/Test Splits (Patient-Aware)
# =====================================

from sklearn.model_selection import train_test_split
import numpy as np


def split_subject_dependent(
    X_all, y_all, patient_ids, target_subject,
    test_size=0.3, random_state=42
):
    """
    Train/test split using data from ONE subject only.
    """

    X_all = np.array(X_all, dtype=object)
    y_all = np.array(y_all)
    patient_ids = np.array(patient_ids)

    mask = patient_ids == target_subject
    X = X_all[mask]
    y = y_all[mask]

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        stratify=y,
        random_state=random_state
    )

    return X_train, X_test, y_train, y_test


def split_population_finetune(
    X_all, y_all, patient_ids, target_subject,
    test_size=0.3, random_state=42
):
    """
    Population training + subject-specific fine-tuning/testing.
    """

    X_all = np.array(X_all, dtype=object)
    y_all = np.array(y_all)
    patient_ids = np.array(patient_ids)

    # Target subject
    mask_target = patient_ids == target_subject
    X_target = X_all[mask_target]
    y_target = y_all[mask_target]

    # Population (all other subjects)
    X_population = X_all[~mask_target]
    y_population = y_all[~mask_target]

    # Split target subject
    X_train_t, X_test_t, y_train_t, y_test_t = train_test_split(
        X_target,
        y_target,
        test_size=test_size,
        stratify=y_target,
        random_state=random_state
    )

    # Train = population + part of target subject
    X_train = np.concatenate([X_population, X_train_t])
    y_train = np.concatenate([y_population, y_train_t])

    return X_train, X_test_t, y_train, y_test_t

In [47]:
# ===============================================
# ⚙️ Baseline Logistic Regression – GLOBAL MODEL
# ===============================================

import numpy as np
import time
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split


# -------------------------------------------------
# Prepare data
# -------------------------------------------------
X = np.array([x.flatten() for x in X_all], dtype=np.float32)
y = np.array(y_all)

print("Total windows:", len(y))
print("Unique classes:", len(np.unique(y)))


# -------------------------------------------------
# Global stratified split (ALL patients, ALL classes)
# -------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    stratify=y,          # ✅ guarantees all classes in train & test
    random_state=42
)

print(f"Train size: {len(y_train)}, Test size: {len(y_test)}")
print("Train class count:", len(np.unique(y_train)))
print("Test class count :", len(np.unique(y_test)))


# -------------------------------------------------
# Encode labels
# -------------------------------------------------
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc  = le.transform(y_test)


# -------------------------------------------------
# Scale features
# -------------------------------------------------
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)


# -------------------------------------------------
# Train Logistic Regression
# -------------------------------------------------
clf = LogisticRegression(
    solver="lbfgs",
    max_iter=2000,
    class_weight="balanced",
    n_jobs=1,
    random_state=42
)

print("\nTraining global Logistic Regression model...")
clf.fit(X_train, y_train_enc)


# -------------------------------------------------
# Inference + timing
# -------------------------------------------------
start = time.perf_counter()
y_pred = clf.predict(X_test)
end = time.perf_counter()

inf_time_ms = (end - start) / len(y_test_enc) * 1000


# -------------------------------------------------
# Metrics
# -------------------------------------------------
acc  = accuracy_score(y_test_enc, y_pred)
prec = precision_score(y_test_enc, y_pred, average="macro", zero_division=0)
rec  = recall_score(y_test_enc, y_pred, average="macro")
f1   = f1_score(y_test_enc, y_pred, average="macro")


# -------------------------------------------------
# Results
# -------------------------------------------------
print("\n📌 Global Logistic Regression Results")
print(f"Accuracy  → {acc:.3f}")
print(f"Precision → {prec:.3f}")
print(f"Recall    → {rec:.3f}")
print(f"F1-score  → {f1:.3f}")
print(f"Inference → avg per window: {inf_time_ms:.3f} ms")

Total windows: 1716
Unique classes: 8
Train size: 1201, Test size: 515
Train class count: 8
Test class count : 8

Training global Logistic Regression model...

📌 Global Logistic Regression Results
Accuracy  → 0.301
Precision → 0.312
Recall    → 0.311
F1-score  → 0.312
Inference → avg per window: 0.079 ms


In [48]:
# ===============================================
# ⚡ XGBoost EEG Classifier – GLOBAL MODEL
# ===============================================

import time
import numpy as np
from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
import xgboost as xgb


# -------------------------------------------------
# Prepare full dataset
# -------------------------------------------------
X_all_np = np.array([x.flatten() for x in X_all], dtype=np.float32)
y_all_np = np.array(y_all)

le = LabelEncoder()
y_all_enc = le.fit_transform(y_all_np)
num_classes = len(le.classes_)

print(f"Classes ({num_classes}):", le.classes_)


# -------------------------------------------------
# Stratified GLOBAL split (ensures all classes)
# -------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_all_np,
    y_all_enc,
    test_size=0.25,
    stratify=y_all_enc,
    random_state=42
)

print(f"Train size: {X_train.shape}, Test size: {X_test.shape}")


# -------------------------------------------------
# Convert to DMatrix
# -------------------------------------------------
dtrain = xgb.DMatrix(X_train, label=y_train)
dtest  = xgb.DMatrix(X_test,  label=y_test)


# -------------------------------------------------
# XGBoost parameters (balanced & fast)
# -------------------------------------------------
params = {
    "objective": "multi:softmax",
    "num_class": num_classes,
    "max_depth": 5,
    "eta": 0.08,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "tree_method": "hist",
    "eval_metric": "mlogloss",
    "seed": 42,
}


# -------------------------------------------------
# Train model
# -------------------------------------------------
print("\n🧠 Training GLOBAL XGBoost model...")
model = xgb.train(
    params=params,
    dtrain=dtrain,
    num_boost_round=250,
    evals=[(dtrain, "train"), (dtest, "test")],
    early_stopping_rounds=20,
    verbose_eval=20
)


# -------------------------------------------------
# Inference timing
# -------------------------------------------------
start = time.perf_counter()
y_pred = model.predict(dtest)
end = time.perf_counter()

inf_time_ms = (end - start) / len(y_test) * 1000


# -------------------------------------------------
# Metrics
# -------------------------------------------------
acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average="macro", zero_division=0)
rec  = recall_score(y_test, y_pred, average="macro")
f1   = f1_score(y_test, y_pred, average="macro")


# -------------------------------------------------
# Results
# -------------------------------------------------
print("\n📌 GLOBAL XGBoost Evaluation")
print(f"Accuracy  → {acc:.3f}")
print(f"Precision → {prec:.3f}")
print(f"Recall    → {rec:.3f}")
print(f"F1-score  → {f1:.3f}")
print(f"Inference → avg per window: {inf_time_ms:.3f} ms")

Classes (8): ['Eyebrow Raise' 'Foot Dorsiflexion' 'Jaw Clench' 'Knee Extension'
 'Left Hand Grasp' 'Lip Purse' 'Right Hand Grasp' 'Shoulder Shrug']
Train size: (1287, 17920), Test size: (429, 17920)

🧠 Training GLOBAL XGBoost model...
[0]	train-mlogloss:1.95800	test-mlogloss:2.03315
[20]	train-mlogloss:0.76007	test-mlogloss:1.58294
[40]	train-mlogloss:0.35372	test-mlogloss:1.44066
[60]	train-mlogloss:0.17876	test-mlogloss:1.36955
[80]	train-mlogloss:0.09626	test-mlogloss:1.33423
[100]	train-mlogloss:0.05738	test-mlogloss:1.31069
[120]	train-mlogloss:0.03771	test-mlogloss:1.29740
[140]	train-mlogloss:0.02734	test-mlogloss:1.29238
[160]	train-mlogloss:0.02108	test-mlogloss:1.28698
[180]	train-mlogloss:0.01721	test-mlogloss:1.28646
[187]	train-mlogloss:0.01619	test-mlogloss:1.28568

📌 GLOBAL XGBoost Evaluation
Accuracy  → 0.501
Precision → 0.523
Recall    → 0.512
F1-score  → 0.513
Inference → avg per window: 0.043 ms


In [49]:
# ==========================================
# 🧠 EEGNet — GLOBAL MODEL EVALUATION
# ==========================================

import numpy as np
import tensorflow as tf
import time
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Conv2D, DepthwiseConv2D, SeparableConv2D,
                                     BatchNormalization, Activation, AveragePooling2D,
                                     Dropout, Flatten, Dense)
from tensorflow.keras.constraints import max_norm
from tensorflow.keras.regularizers import l2


# -------------------------------------------------
# EEGNet definition (unchanged)
# -------------------------------------------------
def EEGNet(nb_classes, Chans, Samples,
           dropoutRate=0.6, kernLength=64, F1=8, D=2,
           norm_rate=0.25, l2_reg=1e-4):

    F2 = F1 * D
    reg = l2(l2_reg)
    inp = Input(shape=(Chans, Samples, 1))

    x = Conv2D(F1, (1, kernLength), padding='same', use_bias=False,
               kernel_regularizer=reg)(inp)
    x = BatchNormalization()(x)

    x = DepthwiseConv2D((Chans, 1), use_bias=False, depth_multiplier=D,
                        depthwise_constraint=max_norm(1.),
                        depthwise_regularizer=reg)(x)
    x = BatchNormalization()(x)
    x = Activation('elu')(x)
    x = AveragePooling2D((1, 4))(x)
    x = Dropout(dropoutRate)(x)

    x = SeparableConv2D(F2, (1, 16), use_bias=False, padding='same',
                        depthwise_regularizer=reg,
                        pointwise_regularizer=reg)(x)
    x = BatchNormalization()(x)
    x = Activation('elu')(x)
    x = AveragePooling2D((1, 8))(x)
    x = Dropout(dropoutRate)(x)

    x = Flatten()(x)
    x = Dense(nb_classes, kernel_constraint=max_norm(norm_rate),
              kernel_regularizer=reg)(x)
    out = Activation('softmax')(x)

    return Model(inp, out)


# -------------------------------------------------
# Prepare GLOBAL dataset
# -------------------------------------------------
X = np.array(X_all, dtype=np.float32)
y = np.array(y_all)

# ensure (N, C, T)
if X.shape[1] != 14:
    X = X.transpose(0, 2, 1)

X = np.expand_dims(X, -1)

le = LabelEncoder()
y_enc = le.fit_transform(y)
num_classes = len(le.classes_)

y_cat = to_categorical(y_enc, num_classes)

print(f"Classes ({num_classes}):", le.classes_)


# -------------------------------------------------
# Stratified GLOBAL split
# -------------------------------------------------
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y_enc,
    test_size=0.25,
    stratify=y_enc,
    random_state=42
)

y_tr_cat = to_categorical(y_tr, num_classes)
y_te_cat = to_categorical(y_te, num_classes)


# -------------------------------------------------
# Class weights (global)
# -------------------------------------------------
cw_vals = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_tr),
    y=y_tr
)
class_weights = dict(enumerate(cw_vals))


# -------------------------------------------------
# Build & train model
# -------------------------------------------------
Chans, Samples = X_tr.shape[1], X_tr.shape[2]
model = EEGNet(num_classes, Chans, Samples)

model.compile(
    optimizer=tf.keras.optimizers.Adam(5e-4),
    loss="categorical_crossentropy"
)

print("\n🧠 Training GLOBAL EEGNet...")
model.fit(
    X_tr, y_tr_cat,
    validation_split=0.1,
    epochs=120,
    batch_size=32,
    class_weight=class_weights,
    verbose=1
)


# -------------------------------------------------
# Inference timing
# -------------------------------------------------
start = time.perf_counter()
y_pred = np.argmax(model.predict(X_te, verbose=0), axis=1)
end = time.perf_counter()

inf_time_ms = (end - start) / len(y_pred) * 1000


# -------------------------------------------------
# Metrics
# -------------------------------------------------
acc  = accuracy_score(y_te, y_pred)
prec = precision_score(y_te, y_pred, average="macro", zero_division=0)
rec  = recall_score(y_te, y_pred, average="macro")
f1   = f1_score(y_te, y_pred, average="macro")


# -------------------------------------------------
# Results
# -------------------------------------------------
print("\n📌 EEGNet GLOBAL Evaluation")
print(f"Accuracy  → {acc:.3f}")
print(f"Precision → {prec:.3f}")
print(f"Recall    → {rec:.3f}")
print(f"F1-score  → {f1:.3f}")
print(f"Inference → avg per window: {inf_time_ms:.3f} ms")

Classes (8): ['Eyebrow Raise' 'Foot Dorsiflexion' 'Jaw Clench' 'Knee Extension'
 'Left Hand Grasp' 'Lip Purse' 'Right Hand Grasp' 'Shoulder Shrug']

🧠 Training GLOBAL EEGNet...
Epoch 1/120
37/37 ━━━━━━━━━━━━━━━━━━━━ 9s 156ms/step - loss: 2.0389 - val_loss: 2.1262
Epoch 2/120
37/37 ━━━━━━━━━━━━━━━━━━━━ 6s 161ms/step - loss: 1.9348 - val_loss: 2.0629
Epoch 3/120
37/37 ━━━━━━━━━━━━━━━━━━━━ 5s 141ms/step - loss: 1.8837 - val_loss: 2.0721
Epoch 4/120
37/37 ━━━━━━━━━━━━━━━━━━━━ 5s 131ms/step - loss: 1.8469 - val_loss: 2.1150
Epoch 5/120
37/37 ━━━━━━━━━━━━━━━━━━━━ 5s 129ms/step - loss: 1.8041 - val_loss: 2.5505
Epoch 6/120
37/37 ━━━━━━━━━━━━━━━━━━━━ 5s 140ms/step - loss: 1.7889 - val_loss: 2.1576
Epoch 7/120
37/37 ━━━━━━━━━━━━━━━━━━━━ 6s 149ms/step - loss: 1.7576 - val_loss: 2.0444
Epoch 8/120
37/37 ━━━━━━━━━━━━━━━━━━━━ 6s 152ms/step - loss: 1.7410 - val_loss: 1.9740
Epoch 9/120
37/37 ━━━━━━━━━━━━━━━━━━━━ 5s 142ms/step - loss: 1.7230 - val_loss: 2.1006
Epoch 10/120
37/37 ━━━━━━━━━━━━━━━━━━━━ 

In [55]:
# ==========================================
# 🧠 EEGNet — GLOBAL MODEL (FINAL, REVIEWER-SAFE)
# ==========================================

import numpy as np
import tensorflow as tf
import time
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Conv2D, DepthwiseConv2D, SeparableConv2D,
                                     BatchNormalization, Activation, AveragePooling2D,
                                     Dropout, Flatten, Dense)
from tensorflow.keras.constraints import max_norm
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping


# -------------------------------------------------
# EEGNet definition (UNCHANGED)
# -------------------------------------------------
def EEGNet(nb_classes, Chans, Samples,
           dropoutRate=0.6, kernLength=64, F1=8, D=2,
           norm_rate=0.25, l2_reg=1e-4):

    F2 = F1 * D
    reg = l2(l2_reg)
    inp = Input(shape=(Chans, Samples, 1))

    x = Conv2D(F1, (1, kernLength), padding='same', use_bias=False,
               kernel_regularizer=reg)(inp)
    x = BatchNormalization()(x)

    x = DepthwiseConv2D((Chans, 1), use_bias=False, depth_multiplier=D,
                        depthwise_constraint=max_norm(1.),
                        depthwise_regularizer=reg)(x)
    x = BatchNormalization()(x)
    x = Activation('elu')(x)
    x = AveragePooling2D((1, 4))(x)
    x = Dropout(dropoutRate)(x)

    x = SeparableConv2D(F2, (1, 16), padding='same', use_bias=False,
                        depthwise_regularizer=reg,
                        pointwise_regularizer=reg)(x)
    x = BatchNormalization()(x)
    x = Activation('elu')(x)
    x = AveragePooling2D((1, 8))(x)
    x = Dropout(dropoutRate)(x)

    x = Flatten()(x)
    x = Dense(nb_classes,
              kernel_constraint=max_norm(norm_rate),
              kernel_regularizer=reg)(x)
    out = Activation('softmax')(x)

    return Model(inp, out)


# -------------------------------------------------
# Prepare GLOBAL dataset
# -------------------------------------------------
X = np.array(X_all, dtype=np.float32)
y = np.array(y_all)

# ensure (N, C, T)
if X.shape[1] != 14:
    X = X.transpose(0, 2, 1)

X = np.expand_dims(X, -1)

le = LabelEncoder()
y_enc = le.fit_transform(y)
num_classes = len(le.classes_)
y_cat = to_categorical(y_enc, num_classes)


# -------------------------------------------------
# Stratified GLOBAL split (TEST SET IS SEALED)
# -------------------------------------------------
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y_enc,
    test_size=0.25,
    stratify=y_enc,
    random_state=42
)

y_tr_cat = to_categorical(y_tr, num_classes)
y_te_cat = to_categorical(y_te, num_classes)


# -------------------------------------------------
# Class weights (GLOBAL)
# -------------------------------------------------
cw_vals = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_tr),
    y=y_tr
)
class_weights = dict(enumerate(cw_vals))


# -------------------------------------------------
# Build model
# -------------------------------------------------
Chans, Samples = X_tr.shape[1], X_tr.shape[2]
model = EEGNet(num_classes, Chans, Samples)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)


# -------------------------------------------------
# Early stopping (VALIDATION FROM TRAINING ONLY)
# -------------------------------------------------
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=20,
    restore_best_weights=True
)


# -------------------------------------------------
# Train model
# -------------------------------------------------
print("\n🧠 Training GLOBAL EEGNet (final)...")
model.fit(
    X_tr, y_tr_cat,
    validation_split=0.1,     # ← ONLY from training data
    epochs=150,
    batch_size=32,
    class_weight=class_weights,
    callbacks=[early_stop],
    verbose=1
)


# -------------------------------------------------
# Inference timing (classifier forward-pass only)
# -------------------------------------------------
start = time.perf_counter()
y_pred = np.argmax(model.predict(X_te, verbose=0), axis=1)
end = time.perf_counter()

inf_time_ms = (end - start) / len(y_pred) * 1000


# -------------------------------------------------
# Metrics (TEST SET — ONCE)
# -------------------------------------------------
acc  = accuracy_score(y_te, y_pred)
prec = precision_score(y_te, y_pred, average="macro", zero_division=0)
rec  = recall_score(y_te, y_pred, average="macro")
f1   = f1_score(y_te, y_pred, average="macro")


# -------------------------------------------------
# Results
# -------------------------------------------------
print("\n📌 EEGNet GLOBAL Evaluation (Final)")
print(f"Accuracy  → {acc:.3f}")
print(f"Precision → {prec:.3f}")
print(f"Recall    → {rec:.3f}")
print(f"F1-score  → {f1:.3f}")
print(f"Inference → avg per window: {inf_time_ms:.3f} ms")


🧠 Training GLOBAL EEGNet (final)...
Epoch 1/150
37/37 ━━━━━━━━━━━━━━━━━━━━ 6s 94ms/step - accuracy: 0.1494 - loss: 2.0767 - val_accuracy: 0.0775 - val_loss: 3.1020
Epoch 2/150
37/37 ━━━━━━━━━━━━━━━━━━━━ 4s 97ms/step - accuracy: 0.1908 - loss: 1.9131 - val_accuracy: 0.0775 - val_loss: 3.4985
Epoch 3/150
37/37 ━━━━━━━━━━━━━━━━━━━━ 3s 85ms/step - accuracy: 0.2176 - loss: 1.8536 - val_accuracy: 0.1628 - val_loss: 2.6109
Epoch 4/150
37/37 ━━━━━━━━━━━━━━━━━━━━ 4s 98ms/step - accuracy: 0.2720 - loss: 1.8054 - val_accuracy: 0.0775 - val_loss: 3.6707
Epoch 5/150
37/37 ━━━━━━━━━━━━━━━━━━━━ 4s 94ms/step - accuracy: 0.2720 - loss: 1.7567 - val_accuracy: 0.0775 - val_loss: 4.3775
Epoch 6/150
37/37 ━━━━━━━━━━━━━━━━━━━━ 3s 91ms/step - accuracy: 0.3247 - loss: 1.7153 - val_accuracy: 0.1318 - val_loss: 2.4054
Epoch 7/150
37/37 ━━━━━━━━━━━━━━━━━━━━ 3s 92ms/step - accuracy: 0.3454 - loss: 1.7071 - val_accuracy: 0.0775 - val_loss: 4.4950
Epoch 8/150
37/37 ━━━━━━━━━━━━━━━━━━━━ 3s 89ms/step - accuracy: 0.3

In [50]:
# ===============================================
# 🧠 EEG-TCNet — GLOBAL MODEL EVALUATION
# ===============================================

import numpy as np
import tensorflow as tf
import time
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Conv2D, DepthwiseConv2D, SeparableConv2D,
                                     BatchNormalization, Activation, AveragePooling2D,
                                     Dropout, Flatten, Dense, Add)
from tensorflow.keras.constraints import max_norm
from tensorflow.keras.regularizers import l2
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


# -------------------------------------------------
# EEG-TCNet definition
# -------------------------------------------------
def EEGTCNet(nb_classes, Chans, Samples,
             n_layers=2, F1=8, D=2, kernel_s=4, kernel_t=32,
             dropout=0.5, dropout_eeg=0.3, activation='elu',
             lr=1e-3, l2_reg=1e-4):

    F2 = F1 * D
    reg = l2(l2_reg)
    inp = Input(shape=(Chans, Samples, 1))

    x = Conv2D(F1, (1, kernel_t), padding='same', use_bias=False,
               kernel_regularizer=reg)(inp)
    x = BatchNormalization()(x)

    x = DepthwiseConv2D((Chans, 1), depth_multiplier=D, use_bias=False,
                        depthwise_constraint=max_norm(1.),
                        depthwise_regularizer=reg)(x)
    x = BatchNormalization()(x)
    x = Activation(activation)(x)
    x = AveragePooling2D((1, 4))(x)
    x = Dropout(dropout_eeg)(x)

    for _ in range(n_layers):
        res = x
        x = SeparableConv2D(F2, (1, kernel_s), padding='same',
                            use_bias=False,
                            depthwise_regularizer=reg,
                            pointwise_regularizer=reg)(x)
        x = BatchNormalization()(x)
        x = Activation(activation)(x)

        x = SeparableConv2D(F2, (1, kernel_s), padding='same',
                            use_bias=False,
                            depthwise_regularizer=reg,
                            pointwise_regularizer=reg)(x)
        x = BatchNormalization()(x)

        x = Add()([x, res])
        x = Activation(activation)(x)
        x = AveragePooling2D((1, 2))(x)
        x = Dropout(dropout)(x)

    x = Flatten()(x)
    x = Dense(nb_classes, kernel_constraint=max_norm(0.25))(x)
    out = Activation('softmax')(x)

    model = Model(inp, out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(lr),
        loss='categorical_crossentropy'
    )
    return model


# -------------------------------------------------
# Prepare GLOBAL dataset
# -------------------------------------------------
X = np.array(X_all, dtype=np.float32)
y = np.array(y_all)

# ensure (N, C, T)
if X.shape[1] != 14:
    X = X.transpose(0, 2, 1)

X = np.expand_dims(X, -1)

le = LabelEncoder()
y_enc = le.fit_transform(y)
num_classes = len(le.classes_)
y_cat = to_categorical(y_enc, num_classes)

print(f"Classes ({num_classes}):", le.classes_)


# -------------------------------------------------
# Stratified GLOBAL split
# -------------------------------------------------
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y_enc,
    test_size=0.25,
    stratify=y_enc,
    random_state=42
)

y_tr_cat = to_categorical(y_tr, num_classes)
y_te_cat = to_categorical(y_te, num_classes)


# -------------------------------------------------
# Class weights
# -------------------------------------------------
cw_vals = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_tr),
    y=y_tr
)
class_weights = dict(enumerate(cw_vals))


# -------------------------------------------------
# Train model
# -------------------------------------------------
Chans, Samples = X_tr.shape[1], X_tr.shape[2]
model = EEGTCNet(num_classes, Chans, Samples)

print("\n🧠 Training GLOBAL EEG-TCNet...")
model.fit(
    X_tr, y_tr_cat,
    validation_split=0.1,
    epochs=100,
    batch_size=32,
    class_weight=class_weights,
    verbose=1
)


# -------------------------------------------------
# Inference timing
# -------------------------------------------------
start = time.perf_counter()
y_pred = np.argmax(model.predict(X_te, verbose=0), axis=1)
end = time.perf_counter()

inf_time_ms = (end - start) / len(y_pred) * 1000


# -------------------------------------------------
# Metrics
# -------------------------------------------------
acc  = accuracy_score(y_te, y_pred)
prec = precision_score(y_te, y_pred, average="macro", zero_division=0)
rec  = recall_score(y_te, y_pred, average="macro")
f1   = f1_score(y_te, y_pred, average="macro")


# -------------------------------------------------
# Results
# -------------------------------------------------
print("\n📌 EEG-TCNet GLOBAL Evaluation")
print(f"Accuracy  → {acc:.3f}")
print(f"Precision → {prec:.3f}")
print(f"Recall    → {rec:.3f}")
print(f"F1-score  → {f1:.3f}")
print(f"Inference → avg per window: {inf_time_ms:.3f} ms")

Classes (8): ['Eyebrow Raise' 'Foot Dorsiflexion' 'Jaw Clench' 'Knee Extension'
 'Left Hand Grasp' 'Lip Purse' 'Right Hand Grasp' 'Shoulder Shrug']

🧠 Training GLOBAL EEG-TCNet...
Epoch 1/100
37/37 ━━━━━━━━━━━━━━━━━━━━ 6s 87ms/step - loss: 2.0343 - val_loss: 9.3156
Epoch 2/100
37/37 ━━━━━━━━━━━━━━━━━━━━ 3s 78ms/step - loss: 1.8163 - val_loss: 10.3374
Epoch 3/100
37/37 ━━━━━━━━━━━━━━━━━━━━ 3s 86ms/step - loss: 1.7641 - val_loss: 6.2450
Epoch 4/100
37/37 ━━━━━━━━━━━━━━━━━━━━ 3s 83ms/step - loss: 1.6951 - val_loss: 9.0126
Epoch 5/100
37/37 ━━━━━━━━━━━━━━━━━━━━ 3s 78ms/step - loss: 1.6622 - val_loss: 2.7743
Epoch 6/100
37/37 ━━━━━━━━━━━━━━━━━━━━ 3s 79ms/step - loss: 1.5950 - val_loss: 5.2157
Epoch 7/100
37/37 ━━━━━━━━━━━━━━━━━━━━ 3s 80ms/step - loss: 1.6208 - val_loss: 8.0370
Epoch 8/100
37/37 ━━━━━━━━━━━━━━━━━━━━ 3s 77ms/step - loss: 1.5622 - val_loss: 7.6849
Epoch 9/100
37/37 ━━━━━━━━━━━━━━━━━━━━ 3s 79ms/step - loss: 1.5813 - val_loss: 3.7889
Epoch 10/100
37/37 ━━━━━━━━━━━━━━━━━━━━ 3s 79

In [57]:
# ===============================================
# 🧠 EEG-TCNet — GLOBAL MODEL (FINAL, REVIEWER-SAFE)
# ===============================================

import numpy as np
import tensorflow as tf
import time
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Conv2D, DepthwiseConv2D, SeparableConv2D,
                                     BatchNormalization, Activation, AveragePooling2D,
                                     Dropout, Flatten, Dense, Add)
from tensorflow.keras.constraints import max_norm
from tensorflow.keras.regularizers import l2
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


# -------------------------------------------------
# EEG-TCNet definition (UNCHANGED)
# -------------------------------------------------
def EEGTCNet(nb_classes, Chans, Samples,
             n_layers=2, F1=8, D=2,
             kernel_s=4, kernel_t=32,
             dropout=0.5, dropout_eeg=0.3,
             activation='elu', l2_reg=1e-4):

    F2 = F1 * D
    reg = l2(l2_reg)
    inp = Input(shape=(Chans, Samples, 1))

    # --- Spatial block ---
    x = Conv2D(F1, (1, kernel_t), padding='same',
               use_bias=False, kernel_regularizer=reg)(inp)
    x = BatchNormalization()(x)

    x = DepthwiseConv2D((Chans, 1), depth_multiplier=D,
                        use_bias=False,
                        depthwise_constraint=max_norm(1.),
                        depthwise_regularizer=reg)(x)
    x = BatchNormalization()(x)
    x = Activation(activation)(x)
    x = AveragePooling2D((1, 4))(x)
    x = Dropout(dropout_eeg)(x)

    # --- Temporal convolutional blocks ---
    for _ in range(n_layers):
        res = x

        x = SeparableConv2D(F2, (1, kernel_s), padding='same',
                            use_bias=False,
                            depthwise_regularizer=reg,
                            pointwise_regularizer=reg)(x)
        x = BatchNormalization()(x)
        x = Activation(activation)(x)

        x = SeparableConv2D(F2, (1, kernel_s), padding='same',
                            use_bias=False,
                            depthwise_regularizer=reg,
                            pointwise_regularizer=reg)(x)
        x = BatchNormalization()(x)

        x = Add()([x, res])
        x = Activation(activation)(x)
        x = AveragePooling2D((1, 2))(x)
        x = Dropout(dropout)(x)

    # --- Classifier ---
    x = Flatten()(x)
    x = Dense(nb_classes,
              kernel_constraint=max_norm(0.25))(x)
    out = Activation('softmax')(x)

    return Model(inp, out)


# -------------------------------------------------
# Prepare GLOBAL dataset
# -------------------------------------------------
X = np.array(X_all, dtype=np.float32)
y = np.array(y_all)

# ensure (N, C, T)
if X.shape[1] != 14:
    X = X.transpose(0, 2, 1)

X = np.expand_dims(X, -1)

le = LabelEncoder()
y_enc = le.fit_transform(y)
num_classes = len(le.classes_)
y_cat = to_categorical(y_enc, num_classes)


# -------------------------------------------------
# Stratified GLOBAL split (TEST SET SEALED)
# -------------------------------------------------
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y_enc,
    test_size=0.25,
    stratify=y_enc,
    random_state=42
)

y_tr_cat = to_categorical(y_tr, num_classes)
y_te_cat = to_categorical(y_te, num_classes)


# -------------------------------------------------
# Class weights (GLOBAL)
# -------------------------------------------------
cw_vals = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_tr),
    y=y_tr
)
class_weights = dict(enumerate(cw_vals))


# -------------------------------------------------
# Build model
# -------------------------------------------------
Chans, Samples = X_tr.shape[1], X_tr.shape[2]
model = EEGTCNet(num_classes, Chans, Samples)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)


# -------------------------------------------------
# Early stopping (VALIDATION FROM TRAINING ONLY)
# -------------------------------------------------
early_stop = EarlyStopping(
    monitor="val_accuracy",
    patience=50,
    restore_best_weights=True
)


# -------------------------------------------------
# Train model
# -------------------------------------------------
print("\n🧠 Training GLOBAL EEG-TCNet (final)...")
model.fit(
    X_tr, y_tr_cat,
    validation_split=0.1,     # ← ONLY from training data
    epochs=150,               # ← SAME as EEGNet
    batch_size=32,
    class_weight=class_weights,
    callbacks=[early_stop],
    verbose=1
)


# -------------------------------------------------
# Inference timing (classifier forward-pass only)
# -------------------------------------------------
start = time.perf_counter()
y_pred = np.argmax(model.predict(X_te, verbose=0), axis=1)
end = time.perf_counter()

inf_time_ms = (end - start) / len(y_pred) * 1000


# -------------------------------------------------
# Metrics (TEST SET — ONCE)
# -------------------------------------------------
acc  = accuracy_score(y_te, y_pred)
prec = precision_score(y_te, y_pred, average="macro", zero_division=0)
rec  = recall_score(y_te, y_pred, average="macro")
f1   = f1_score(y_te, y_pred, average="macro")


# -------------------------------------------------
# Results
# -------------------------------------------------
print("\n📌 EEG-TCNet GLOBAL Evaluation (Final)")
print(f"Accuracy  → {acc:.3f}")
print(f"Precision → {prec:.3f}")
print(f"Recall    → {rec:.3f}")
print(f"F1-score  → {f1:.3f}")
print(f"Inference → avg per window: {inf_time_ms:.3f} ms")



🧠 Training GLOBAL EEG-TCNet (final)...
Epoch 1/150
37/37 ━━━━━━━━━━━━━━━━━━━━ 6s 88ms/step - accuracy: 0.1986 - loss: 2.0420 - val_accuracy: 0.1318 - val_loss: 4.2605
Epoch 2/150
37/37 ━━━━━━━━━━━━━━━━━━━━ 3s 76ms/step - accuracy: 0.2565 - loss: 1.8913 - val_accuracy: 0.1318 - val_loss: 3.1680
Epoch 3/150
37/37 ━━━━━━━━━━━━━━━━━━━━ 3s 73ms/step - accuracy: 0.2720 - loss: 1.8290 - val_accuracy: 0.1318 - val_loss: 4.8064
Epoch 4/150
37/37 ━━━━━━━━━━━━━━━━━━━━ 3s 77ms/step - accuracy: 0.3040 - loss: 1.7492 - val_accuracy: 0.1318 - val_loss: 3.8597
Epoch 5/150
37/37 ━━━━━━━━━━━━━━━━━━━━ 3s 81ms/step - accuracy: 0.3100 - loss: 1.7148 - val_accuracy: 0.0775 - val_loss: 8.9774
Epoch 6/150
37/37 ━━━━━━━━━━━━━━━━━━━━ 3s 74ms/step - accuracy: 0.3368 - loss: 1.6931 - val_accuracy: 0.0930 - val_loss: 18.5230
Epoch 7/150
37/37 ━━━━━━━━━━━━━━━━━━━━ 3s 75ms/step - accuracy: 0.3402 - loss: 1.6174 - val_accuracy: 0.0930 - val_loss: 16.9128
Epoch 8/150
37/37 ━━━━━━━━━━━━━━━━━━━━ 3s 77ms/step - accuracy

In [51]:
# ==========================================================
# ⚙️ GLOBAL Filter-Bank Riemann + Tangent Space + LogReg
# ==========================================================

import numpy as np, time, warnings
warnings.filterwarnings("ignore")

from pyriemann.estimation import Covariances
from pyriemann.tangentspace import TangentSpace
from scipy.signal import butter, filtfilt
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# -------------------------------------------------
# 1) Global data
# -------------------------------------------------
X = np.asarray(X_all, dtype=np.float32)
y = np.asarray(y_all)

def ensure_nct(X):
    n, a, b = X.shape
    if a <= 64 and b >= 50: return X
    if b <= 64 and a >= 50: return X.transpose(0,2,1)
    return X

X = ensure_nct(X)
X = X - X.mean(axis=2, keepdims=True)  # de-mean per trial

le = LabelEncoder()
y_enc = le.fit_transform(y)
n_classes = len(le.classes_)

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y_enc,
    test_size=0.25,
    stratify=y_enc,
    random_state=42
)

print(f"Train shape: {X_tr.shape}, Test shape: {X_te.shape}")
print("Classes:", le.classes_)

# -------------------------------------------------
# 2) Filter bank
# -------------------------------------------------
FS = 256.0
bands = [
    (8, 12),    # μ
    (13, 20),   # low-β
    (20, 30),   # high-β
    (4, 7),     # θ
]

def bandpass(data, lo, hi, fs=FS, order=4):
    b, a = butter(order, [lo/(fs/2), hi/(fs/2)], btype='band')
    return filtfilt(b, a, data, axis=2)

# -------------------------------------------------
# 3) Filter-bank Riemann TS features
# -------------------------------------------------
def fb_riemann_ts(Xtr, Xte, ytr, bands):
    feats_tr, feats_te = [], []

    for lo, hi in bands:
        Xtr_f = bandpass(Xtr, lo, hi)
        Xte_f = bandpass(Xte, lo, hi)

        cov_tr = Covariances(estimator="oas").fit_transform(Xtr_f)
        cov_te = Covariances(estimator="oas").transform(Xte_f)

        ts = TangentSpace()
        ts.fit(cov_tr, ytr)

        feats_tr.append(ts.transform(cov_tr))
        feats_te.append(ts.transform(cov_te))

    return np.concatenate(feats_tr, axis=1), np.concatenate(feats_te, axis=1)

print("Building filter-bank tangent-space features...")
Xtr_ts, Xte_ts = fb_riemann_ts(X_tr, X_te, y_tr, bands)

scaler = StandardScaler()
Xtr_ts = scaler.fit_transform(Xtr_ts)
Xte_ts = scaler.transform(Xte_ts)

print(f"TS features: train {Xtr_ts.shape}, test {Xte_ts.shape}")

# -------------------------------------------------
# 4) Logistic Regression (CV on C)
# -------------------------------------------------
param_grid = {"C": np.logspace(-2, 2, 7)}
base_clf = LogisticRegression(
    solver="lbfgs",
    max_iter=2000,
    class_weight="balanced",
    n_jobs=1,
    multi_class="auto",
    random_state=42
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
clf = GridSearchCV(base_clf, param_grid, cv=cv, n_jobs=1, refit=True)

print("Training Logistic Regression (CV on C)...")
clf.fit(Xtr_ts, y_tr)
print("Best C:", clf.best_params_["C"])

# -------------------------------------------------
# 5) Evaluation + inference timing
# -------------------------------------------------
t0 = time.perf_counter()
y_pred = clf.predict(Xte_ts)
t1 = time.perf_counter()

inf_time_ms = (t1 - t0) / len(y_pred) * 1000

acc  = accuracy_score(y_te, y_pred)
prec = precision_score(y_te, y_pred, average="macro", zero_division=0)
rec  = recall_score(y_te, y_pred, average="macro")
f1   = f1_score(y_te, y_pred, average="macro")

print("\n📌 Filter-Bank Riemann + TS + LogReg (GLOBAL)")
print(f"Accuracy  → {acc:.3f}")
print(f"Precision → {prec:.3f}")
print(f"Recall    → {rec:.3f}")
print(f"F1-score  → {f1:.3f}")
print(f"Inference → avg per window: {inf_time_ms:.3f} ms")


Train shape: (1287, 14, 1280), Test shape: (429, 14, 1280)
Classes: ['Eyebrow Raise' 'Foot Dorsiflexion' 'Jaw Clench' 'Knee Extension'
 'Left Hand Grasp' 'Lip Purse' 'Right Hand Grasp' 'Shoulder Shrug']
Building filter-bank tangent-space features...
TS features: train (1287, 420), test (429, 420)
Training Logistic Regression (CV on C)...
Best C: 21.54434690031882

📌 Filter-Bank Riemann + TS + LogReg (GLOBAL)
Accuracy  → 0.830
Precision → 0.835
Recall    → 0.831
F1-score  → 0.831
Inference → avg per window: 0.003 ms


In [52]:
# ==========================================================
# ⚙️ GLOBAL Filter-Bank Riemann + Tangent Space + SVM
# ==========================================================

import numpy as np, time, warnings
warnings.filterwarnings("ignore")

from pyriemann.estimation import Covariances
from pyriemann.tangentspace import TangentSpace
from scipy.signal import butter, filtfilt
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# -------------------------------------------------
# 1) Global data
# -------------------------------------------------
X = np.asarray(X_all, dtype=np.float32)
y = np.asarray(y_all)

def ensure_nct(X):
    n, a, b = X.shape
    if a <= 64 and b >= 50: return X
    if b <= 64 and a >= 50: return X.transpose(0,2,1)
    return X

X = ensure_nct(X)
X = X - X.mean(axis=2, keepdims=True)  # per-trial de-mean

le = LabelEncoder()
y_enc = le.fit_transform(y)
n_classes = len(le.classes_)

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y_enc,
    test_size=0.25,
    stratify=y_enc,
    random_state=42
)

print(f"Train shape: {X_tr.shape}, Test shape: {X_te.shape}")
print("Classes:", le.classes_)

# -------------------------------------------------
# 2) Filter bank
# -------------------------------------------------
FS = 256.0
bands = [
    (8, 12),    # μ
    (13, 20),   # low-β
    (20, 30),   # high-β
    (4, 7),     # θ
]

def bandpass(data, lo, hi, fs=FS, order=4):
    b, a = butter(order, [lo/(fs/2), hi/(fs/2)], btype='band')
    return filtfilt(b, a, data, axis=2)

# -------------------------------------------------
# 3) Filter-bank Riemann TS features
# -------------------------------------------------
def fb_riemann_ts(Xtr, Xte, ytr, bands):
    feats_tr, feats_te = [], []

    for lo, hi in bands:
        Xtr_f = bandpass(Xtr, lo, hi)
        Xte_f = bandpass(Xte, lo, hi)

        cov_tr = Covariances(estimator="oas").fit_transform(Xtr_f)
        cov_te = Covariances(estimator="oas").transform(Xte_f)

        ts = TangentSpace()
        ts.fit(cov_tr, ytr)

        feats_tr.append(ts.transform(cov_tr))
        feats_te.append(ts.transform(cov_te))

    return np.concatenate(feats_tr, axis=1), np.concatenate(feats_te, axis=1)

print("Building filter-bank tangent-space features...")
Xtr_ts, Xte_ts = fb_riemann_ts(X_tr, X_te, y_tr, bands)

scaler = StandardScaler()
Xtr_ts = scaler.fit_transform(Xtr_ts)
Xte_ts = scaler.transform(Xte_ts)

print(f"TS features: train {Xtr_ts.shape}, test {Xte_ts.shape}")

# -------------------------------------------------
# 4) SVM with CV (train only)
# -------------------------------------------------
param_grid = [
    {"kernel": ["linear"], "C": np.logspace(-2, 2, 7)},
    {"kernel": ["rbf"], "C": np.logspace(-2, 2, 7), "gamma": np.logspace(-3, 1, 5)},
]

base_clf = SVC(class_weight="balanced")
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

clf = GridSearchCV(
    base_clf,
    param_grid=param_grid,
    cv=cv,
    n_jobs=1,
    refit=True
)

print("Training SVM (CV on kernel / C / gamma)...")
clf.fit(Xtr_ts, y_tr)
print("Best params:", clf.best_params_)

# -------------------------------------------------
# 5) Evaluation + inference timing
# -------------------------------------------------
t0 = time.perf_counter()
y_pred = clf.predict(Xte_ts)
t1 = time.perf_counter()

inf_time_ms = (t1 - t0) / len(y_pred) * 1000

acc  = accuracy_score(y_te, y_pred)
prec = precision_score(y_te, y_pred, average="macro", zero_division=0)
rec  = recall_score(y_te, y_pred, average="macro")
f1   = f1_score(y_te, y_pred, average="macro")

print("\n📌 Filter-Bank Riemann + TS + SVM (GLOBAL)")
print(f"Accuracy  → {acc:.3f}")
print(f"Precision → {prec:.3f}")
print(f"Recall    → {rec:.3f}")
print(f"F1-score  → {f1:.3f}")
print(f"Inference → avg per window: {inf_time_ms:.3f} ms")


Train shape: (1287, 14, 1280), Test shape: (429, 14, 1280)
Classes: ['Eyebrow Raise' 'Foot Dorsiflexion' 'Jaw Clench' 'Knee Extension'
 'Left Hand Grasp' 'Lip Purse' 'Right Hand Grasp' 'Shoulder Shrug']
Building filter-bank tangent-space features...
TS features: train (1287, 420), test (429, 420)
Training SVM (CV on kernel / C / gamma)...
Best params: {'C': 4.6415888336127775, 'gamma': 0.001, 'kernel': 'rbf'}

📌 Filter-Bank Riemann + TS + SVM (GLOBAL)
Accuracy  → 0.928
Precision → 0.932
Recall    → 0.925
F1-score  → 0.927
Inference → avg per window: 0.294 ms


In [7]:
# ==========================================================
# 🧪 Population → Subject Calibration → Subject Test (ONE)
# ==========================================================

import numpy as np, time, warnings
warnings.filterwarnings("ignore")

from pyriemann.estimation import Covariances
from pyriemann.tangentspace import TangentSpace
from scipy.signal import butter, filtfilt
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, GridSearchCV, train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# -------------------------------------------------
# CONFIG
# -------------------------------------------------
TARGET_SUBJECT = "p1"        # change later in loop
CALIBRATION_RATIO = 0.25
FS = 256.0

bands = [
    (8, 12),   # μ
    (13, 20),  # low-β
    (20, 30),  # high-β
    (4, 7),    # θ
]

# -------------------------------------------------
# Helpers
# -------------------------------------------------
def ensure_nct(X):
    n, a, b = X.shape
    if a <= 64 and b >= 50: return X
    if b <= 64 and a >= 50: return X.transpose(0,2,1)
    return X

def bandpass(data, lo, hi, fs=FS, order=4):
    b, a = butter(order, [lo/(fs/2), hi/(fs/2)], btype='band')
    return filtfilt(b, a, data, axis=2)

def fb_riemann_ts(Xtr, Xte, ytr, bands):
    feats_tr, feats_te = [], []
    for lo, hi in bands:
        Xtr_f = bandpass(Xtr, lo, hi)
        Xte_f = bandpass(Xte, lo, hi)

        cov_tr = Covariances("oas").fit_transform(Xtr_f)
        cov_te = Covariances("oas").transform(Xte_f)

        ts = TangentSpace()
        ts.fit(cov_tr, ytr)

        feats_tr.append(ts.transform(cov_tr))
        feats_te.append(ts.transform(cov_te))

    return np.concatenate(feats_tr, axis=1), np.concatenate(feats_te, axis=1)

def split_population_calibration_test(
    X_all, y_all, patient_ids, target_subject,
    calibration_ratio=0.25, random_state=42
):
    X_all = np.asarray(X_all, dtype=object)
    y_all = np.asarray(y_all)
    patient_ids = np.asarray(patient_ids)

    mask_target = patient_ids == target_subject

    # Population (no target subject)
    X_pop = X_all[~mask_target]
    y_pop = y_all[~mask_target]

    # Target subject → calibration + test
    X_target = X_all[mask_target]
    y_target = y_all[mask_target]

    X_cal, X_test, y_cal, y_test = train_test_split(
        X_target, y_target,
        test_size=1 - calibration_ratio,
        stratify=y_target,
        random_state=random_state
    )

    return X_pop, y_pop, X_cal, y_cal, X_test, y_test

# -------------------------------------------------
# 1) Population / calibration / test split
# -------------------------------------------------
X_pop, y_pop, X_cal, y_cal, X_test, y_test = split_population_calibration_test(
    X_all, y_all, patient_ids,
    TARGET_SUBJECT,
    calibration_ratio=CALIBRATION_RATIO
)

# -------------------------------------------------
# 2) Encode labels (GLOBAL)
# -------------------------------------------------
le = LabelEncoder()
y_pop  = le.fit_transform(y_pop)
y_cal  = le.transform(y_cal)
y_test = le.transform(y_test)

# -------------------------------------------------
# 3) Shape + de-mean
# -------------------------------------------------
X_pop  = ensure_nct(np.asarray(X_pop, dtype=np.float32))
X_cal  = ensure_nct(np.asarray(X_cal, dtype=np.float32))
X_test = ensure_nct(np.asarray(X_test, dtype=np.float32))

X_pop  -= X_pop.mean(axis=2, keepdims=True)
X_cal  -= X_cal.mean(axis=2, keepdims=True)
X_test -= X_test.mean(axis=2, keepdims=True)

print("Population:", X_pop.shape)
print("Calibration:", X_cal.shape)
print("Test:", X_test.shape)

# -------------------------------------------------
# 4) Feature extraction
# -------------------------------------------------
print("Extracting FB-Riemann TS features...")
Xpop_ts, Xcal_ts = fb_riemann_ts(X_pop, X_cal, y_pop, bands)
_, Xtest_ts      = fb_riemann_ts(X_pop, X_test, y_pop, bands)

# -------------------------------------------------
# 5) Calibration-aware normalization
# -------------------------------------------------
scaler = StandardScaler()
scaler.fit(np.vstack([Xpop_ts, Xcal_ts]))

Xpop_ts  = scaler.transform(Xpop_ts)
Xcal_ts  = scaler.transform(Xcal_ts)
Xtest_ts = scaler.transform(Xtest_ts)

# -------------------------------------------------
# 6) Train SVM (population + calibration)
# -------------------------------------------------
Xtrain = np.vstack([Xpop_ts, Xcal_ts])
ytrain = np.concatenate([y_pop, y_cal])

param_grid = [
    {"kernel": ["linear"], "C": np.logspace(-2, 2, 7)},
    {"kernel": ["rbf"], "C": np.logspace(-2, 2, 7), "gamma": np.logspace(-3, 1, 5)},
]

clf = GridSearchCV(
    SVC(class_weight="balanced"),
    param_grid,
    cv=StratifiedKFold(5, shuffle=True, random_state=42),
    n_jobs=1,
    refit=True
)

print("Training calibrated SVM...")
clf.fit(Xtrain, ytrain)
print("Best params:", clf.best_params_)

# -------------------------------------------------
# 7) Evaluation (REALISTIC deployment)
# -------------------------------------------------
t0 = time.perf_counter()
y_pred = clf.predict(Xtest_ts)
t1 = time.perf_counter()

inf_ms = (t1 - t0) / len(y_pred) * 1000

print("\n📌 Population → Calibration → Test (ONE subject)")
print(f"Accuracy  → {accuracy_score(y_test, y_pred):.3f}")
print(f"Precision → {precision_score(y_test, y_pred, average='macro', zero_division=0):.3f}")
print(f"Recall    → {recall_score(y_test, y_pred, average='macro'):.3f}")
print(f"F1-score  → {f1_score(y_test, y_pred, average='macro'):.3f}")
print(f"Inference → {inf_ms:.3f} ms / window")


Population: (1520, 14, 1280)
Calibration: (49, 14, 1280)
Test: (147, 14, 1280)
Extracting FB-Riemann TS features...
Training calibrated SVM...
Best params: {'C': 4.6415888336127775, 'gamma': 0.001, 'kernel': 'rbf'}

📌 Population → Calibration → Test (ONE subject)
Accuracy  → 0.680
Precision → 0.688
Recall    → 0.675
F1-score  → 0.669
Inference → 0.619 ms / window


# Leave One Subject Out Training (Calibration)

In [8]:
# ==========================================================
# 🔁 LOSO — Population → 50% Calibration → Test (ALL patients)
# ==========================================================

import numpy as np, time, warnings
warnings.filterwarnings("ignore")

from pyriemann.estimation import Covariances
from pyriemann.tangentspace import TangentSpace
from scipy.signal import butter, filtfilt
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, GridSearchCV, train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# -------------------------------------------------
# CONFIG
# -------------------------------------------------
CALIBRATION_RATIO = 0.50
FS = 256.0
RANDOM_STATE = 42

bands = [
    (8, 12),   # μ
    (13, 20),  # low-β
    (20, 30),  # high-β
    (4, 7),    # θ
]

# -------------------------------------------------
# Helpers
# -------------------------------------------------
def ensure_nct(X):
    n, a, b = X.shape
    if a <= 64 and b >= 50: return X
    if b <= 64 and a >= 50: return X.transpose(0,2,1)
    return X

def bandpass(data, lo, hi, fs=FS, order=4):
    b, a = butter(order, [lo/(fs/2), hi/(fs/2)], btype='band')
    return filtfilt(b, a, data, axis=2)

def fb_riemann_ts(Xtr, Xte, ytr, bands):
    feats_tr, feats_te = [], []
    for lo, hi in bands:
        Xtr_f = bandpass(Xtr, lo, hi)
        Xte_f = bandpass(Xte, lo, hi)

        cov_tr = Covariances("oas").fit_transform(Xtr_f)
        cov_te = Covariances("oas").transform(Xte_f)

        ts = TangentSpace()
        ts.fit(cov_tr, ytr)

        feats_tr.append(ts.transform(cov_tr))
        feats_te.append(ts.transform(cov_te))

    return np.concatenate(feats_tr, axis=1), np.concatenate(feats_te, axis=1)

def split_population_calibration_test(
    X_all, y_all, patient_ids, target_subject,
    calibration_ratio=0.5, random_state=42
):
    X_all = np.asarray(X_all, dtype=object)
    y_all = np.asarray(y_all)
    patient_ids = np.asarray(patient_ids)

    mask_target = patient_ids == target_subject

    X_pop = X_all[~mask_target]
    y_pop = y_all[~mask_target]

    X_target = X_all[mask_target]
    y_target = y_all[mask_target]

    X_cal, X_test, y_cal, y_test = train_test_split(
        X_target, y_target,
        test_size=1 - calibration_ratio,
        stratify=y_target,
        random_state=random_state
    )

    return X_pop, y_pop, X_cal, y_cal, X_test, y_test

# -------------------------------------------------
# Loop over all patients
# -------------------------------------------------
unique_patients = np.unique(patient_ids)
results = []

print(f"Running LOSO with {int(CALIBRATION_RATIO*100)}% calibration\n")

for pid in unique_patients:
    print(f"▶ Subject {pid}")

    # -----------------------------
    # Split
    # -----------------------------
    X_pop, y_pop, X_cal, y_cal, X_test, y_test = split_population_calibration_test(
        X_all, y_all, patient_ids,
        pid,
        calibration_ratio=CALIBRATION_RATIO,
        random_state=RANDOM_STATE
    )

    # -----------------------------
    # Encode labels (GLOBAL)
    # -----------------------------
    le = LabelEncoder()
    y_pop  = le.fit_transform(y_pop)
    y_cal  = le.transform(y_cal)
    y_test = le.transform(y_test)

    # -----------------------------
    # Shape + de-mean
    # -----------------------------
    X_pop  = ensure_nct(np.asarray(X_pop, dtype=np.float32))
    X_cal  = ensure_nct(np.asarray(X_cal, dtype=np.float32))
    X_test = ensure_nct(np.asarray(X_test, dtype=np.float32))

    X_pop  -= X_pop.mean(axis=2, keepdims=True)
    X_cal  -= X_cal.mean(axis=2, keepdims=True)
    X_test -= X_test.mean(axis=2, keepdims=True)

    # -----------------------------
    # Feature extraction
    # -----------------------------
    Xpop_ts, Xcal_ts = fb_riemann_ts(X_pop, X_cal, y_pop, bands)
    _, Xtest_ts      = fb_riemann_ts(X_pop, X_test, y_pop, bands)

    # -----------------------------
    # Normalization (calibration-aware)
    # -----------------------------
    scaler = StandardScaler()
    scaler.fit(np.vstack([Xpop_ts, Xcal_ts]))

    Xpop_ts  = scaler.transform(Xpop_ts)
    Xcal_ts  = scaler.transform(Xcal_ts)
    Xtest_ts = scaler.transform(Xtest_ts)

    # -----------------------------
    # Train SVM
    # -----------------------------
    Xtrain = np.vstack([Xpop_ts, Xcal_ts])
    ytrain = np.concatenate([y_pop, y_cal])

    clf = GridSearchCV(
        SVC(class_weight="balanced"),
        [
            {"kernel": ["linear"], "C": np.logspace(-2, 2, 7)},
            {"kernel": ["rbf"], "C": np.logspace(-2, 2, 7), "gamma": np.logspace(-3, 1, 5)},
        ],
        cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE),
        n_jobs=1,
        refit=True
    )

    clf.fit(Xtrain, ytrain)

    # -----------------------------
    # Evaluation
    # -----------------------------
    t0 = time.perf_counter()
    y_pred = clf.predict(Xtest_ts)
    t1 = time.perf_counter()

    inf_ms = (t1 - t0) / len(y_pred) * 1000

    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average="macro", zero_division=0)
    rec  = recall_score(y_test, y_pred, average="macro")
    f1   = f1_score(y_test, y_pred, average="macro")

    results.append([acc, prec, rec, f1, inf_ms])

    print(f"  Acc={acc:.3f} | F1={f1:.3f} | Inf={inf_ms:.3f} ms")

# -------------------------------------------------
# Aggregate results
# -------------------------------------------------
results = np.array(results)

print("\n📊 LOSO Results (50% Calibration)")
print(f"Accuracy  : {results[:,0].mean():.3f} ± {results[:,0].std():.3f}")
print(f"Precision : {results[:,1].mean():.3f} ± {results[:,1].std():.3f}")
print(f"Recall    : {results[:,2].mean():.3f} ± {results[:,2].std():.3f}")
print(f"F1-score  : {results[:,3].mean():.3f} ± {results[:,3].std():.3f}")
print(f"Inference : {results[:,4].mean():.3f} ± {results[:,4].std():.3f} ms")


Running LOSO with 50% calibration

▶ Subject p1
  Acc=0.888 | F1=0.887 | Inf=1.552 ms
▶ Subject p10
  Acc=0.972 | F1=0.591 | Inf=0.398 ms
▶ Subject p11
  Acc=0.988 | F1=0.987 | Inf=0.571 ms
▶ Subject p2
  Acc=0.932 | F1=0.932 | Inf=0.356 ms
▶ Subject p3
  Acc=0.961 | F1=0.960 | Inf=0.405 ms
▶ Subject p4
  Acc=0.932 | F1=0.928 | Inf=0.348 ms
▶ Subject p5
  Acc=0.955 | F1=0.956 | Inf=0.435 ms
▶ Subject p6
  Acc=0.936 | F1=0.935 | Inf=0.928 ms
▶ Subject p7
  Acc=0.851 | F1=0.681 | Inf=1.256 ms
▶ Subject p8
  Acc=0.936 | F1=0.938 | Inf=1.539 ms
▶ Subject p9
  Acc=0.854 | F1=0.848 | Inf=0.359 ms

📊 LOSO Results (50% Calibration)
Accuracy  : 0.928 ± 0.043
Precision : 0.883 ± 0.117
Recall    : 0.875 ± 0.122
F1-score  : 0.877 ± 0.120
Inference : 0.741 ± 0.467 ms


In [9]:
# ==========================================================
# 🔁 LOSO — Population → 50% Calibration → Test (LOGISTIC REG)
# ==========================================================

import numpy as np, time, warnings
warnings.filterwarnings("ignore")

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# -------------------------------------------------
# CONFIG
# -------------------------------------------------
CALIBRATION_RATIO = 0.50
RANDOM_STATE = 42

# -------------------------------------------------
# Helpers
# -------------------------------------------------
def split_population_calibration_test(
    X_all, y_all, patient_ids, target_subject,
    calibration_ratio=0.5, random_state=42
):
    X_all = np.asarray(X_all, dtype=object)
    y_all = np.asarray(y_all)
    patient_ids = np.asarray(patient_ids)

    mask_target = patient_ids == target_subject

    X_pop = X_all[~mask_target]
    y_pop = y_all[~mask_target]

    X_target = X_all[mask_target]
    y_target = y_all[mask_target]

    X_cal, X_test, y_cal, y_test = train_test_split(
        X_target, y_target,
        test_size=1 - calibration_ratio,
        stratify=y_target,
        random_state=random_state
    )

    return X_pop, y_pop, X_cal, y_cal, X_test, y_test

# -------------------------------------------------
# Loop over all patients
# -------------------------------------------------
unique_patients = np.unique(patient_ids)
results = []

print(f"Running LOSO Logistic Regression with {int(CALIBRATION_RATIO*100)}% calibration\n")

for pid in unique_patients:
    print(f"▶ Subject {pid}")

    # -----------------------------
    # Split
    # -----------------------------
    X_pop, y_pop, X_cal, y_cal, X_test, y_test = split_population_calibration_test(
        X_all, y_all, patient_ids,
        pid,
        calibration_ratio=CALIBRATION_RATIO,
        random_state=RANDOM_STATE
    )

    # -----------------------------
    # Flatten windows
    # -----------------------------
    X_pop  = np.asarray([x.flatten() for x in X_pop],  dtype=np.float32)
    X_cal  = np.asarray([x.flatten() for x in X_cal],  dtype=np.float32)
    X_test = np.asarray([x.flatten() for x in X_test], dtype=np.float32)

    # -----------------------------
    # Encode labels
    # -----------------------------
    le = LabelEncoder()
    y_pop  = le.fit_transform(y_pop)
    y_cal  = le.transform(y_cal)
    y_test = le.transform(y_test)

    # -----------------------------
    # Scale (population + calibration)
    # -----------------------------
    scaler = StandardScaler()
    scaler.fit(np.vstack([X_pop, X_cal]))

    X_pop  = scaler.transform(X_pop)
    X_cal  = scaler.transform(X_cal)
    X_test = scaler.transform(X_test)

    # -----------------------------
    # Train Logistic Regression
    # -----------------------------
    X_train = np.vstack([X_pop, X_cal])
    y_train = np.concatenate([y_pop, y_cal])

    clf = LogisticRegression(
        solver="lbfgs",
        max_iter=3000,
        class_weight="balanced",
        n_jobs=1,
        random_state=RANDOM_STATE
    )

    clf.fit(X_train, y_train)

    # -----------------------------
    # Evaluation
    # -----------------------------
    t0 = time.perf_counter()
    y_pred = clf.predict(X_test)
    t1 = time.perf_counter()

    inf_ms = (t1 - t0) / len(y_pred) * 1000

    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average="macro", zero_division=0)
    rec  = recall_score(y_test, y_pred, average="macro")
    f1   = f1_score(y_test, y_pred, average="macro")

    results.append([acc, prec, rec, f1, inf_ms])

    print(f"  Acc={acc:.3f} | F1={f1:.3f} | Inf={inf_ms:.3f} ms")

# -------------------------------------------------
# Aggregate results
# -------------------------------------------------
results = np.array(results)

print("\n📊 LOSO Logistic Regression Results (50% Calibration)")
print(f"Accuracy  : {results[:,0].mean():.3f} ± {results[:,0].std():.3f}")
print(f"Precision : {results[:,1].mean():.3f} ± {results[:,1].std():.3f}")
print(f"Recall    : {results[:,2].mean():.3f} ± {results[:,2].std():.3f}")
print(f"F1-score  : {results[:,3].mean():.3f} ± {results[:,3].std():.3f}")
print(f"Inference : {results[:,4].mean():.3f} ± {results[:,4].std():.3f} ms")


Running LOSO Logistic Regression with 50% calibration

▶ Subject p1
  Acc=0.184 | F1=0.097 | Inf=0.079 ms
▶ Subject p10
  Acc=0.347 | F1=0.167 | Inf=0.070 ms
▶ Subject p11
  Acc=0.237 | F1=0.094 | Inf=0.068 ms
▶ Subject p2
  Acc=0.295 | F1=0.171 | Inf=0.065 ms
▶ Subject p3
  Acc=0.196 | F1=0.101 | Inf=0.094 ms
▶ Subject p4
  Acc=0.295 | F1=0.176 | Inf=0.072 ms
▶ Subject p5
  Acc=0.169 | F1=0.096 | Inf=0.070 ms
▶ Subject p6
  Acc=0.213 | F1=0.087 | Inf=0.058 ms
▶ Subject p7
  Acc=0.126 | F1=0.078 | Inf=0.070 ms
▶ Subject p8
  Acc=0.154 | F1=0.089 | Inf=0.075 ms
▶ Subject p9
  Acc=0.293 | F1=0.188 | Inf=0.075 ms

📊 LOSO Logistic Regression Results (50% Calibration)
Accuracy  : 0.228 ± 0.068
Precision : 0.177 ± 0.061
Recall    : 0.104 ± 0.037
F1-score  : 0.122 ± 0.041
Inference : 0.072 ± 0.009 ms


In [10]:
# ==========================================================
# 🔁 LOSO — Population → 50% Calibration → Test (XGBOOST)
# ==========================================================

import numpy as np, time, warnings
warnings.filterwarnings("ignore")

import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# -------------------------------------------------
# CONFIG
# -------------------------------------------------
CALIBRATION_RATIO = 0.50
RANDOM_STATE = 42

# -------------------------------------------------
# Helper
# -------------------------------------------------
def split_population_calibration_test(
    X_all, y_all, patient_ids, target_subject,
    calibration_ratio=0.5, random_state=42
):
    X_all = np.asarray(X_all, dtype=object)
    y_all = np.asarray(y_all)
    patient_ids = np.asarray(patient_ids)

    mask_target = patient_ids == target_subject

    X_pop = X_all[~mask_target]
    y_pop = y_all[~mask_target]

    X_target = X_all[mask_target]
    y_target = y_all[mask_target]

    X_cal, X_test, y_cal, y_test = train_test_split(
        X_target, y_target,
        test_size=1 - calibration_ratio,
        stratify=y_target,
        random_state=random_state
    )

    return X_pop, y_pop, X_cal, y_cal, X_test, y_test

# -------------------------------------------------
# Loop over all patients
# -------------------------------------------------
unique_patients = np.unique(patient_ids)
results = []

print(f"Running LOSO XGBoost with {int(CALIBRATION_RATIO*100)}% calibration\n")

for pid in unique_patients:
    print(f"▶ Subject {pid}")

    # -----------------------------
    # Split
    # -----------------------------
    X_pop, y_pop, X_cal, y_cal, X_test, y_test = split_population_calibration_test(
        X_all, y_all, patient_ids,
        pid,
        calibration_ratio=CALIBRATION_RATIO,
        random_state=RANDOM_STATE
    )

    # -----------------------------
    # Flatten windows
    # -----------------------------
    X_pop  = np.asarray([x.flatten() for x in X_pop],  dtype=np.float32)
    X_cal  = np.asarray([x.flatten() for x in X_cal],  dtype=np.float32)
    X_test = np.asarray([x.flatten() for x in X_test], dtype=np.float32)

    # -----------------------------
    # Encode labels (GLOBAL)
    # -----------------------------
    le = LabelEncoder()
    y_pop  = le.fit_transform(y_pop)
    y_cal  = le.transform(y_cal)
    y_test = le.transform(y_test)

    num_classes = len(le.classes_)

    # -----------------------------
    # Build DMatrix
    # -----------------------------
    dtrain = xgb.DMatrix(
        np.vstack([X_pop, X_cal]),
        label=np.concatenate([y_pop, y_cal])
    )
    dtest = xgb.DMatrix(X_test, label=y_test)

    # -----------------------------
    # XGBoost params
    # -----------------------------
    params = {
        "objective": "multi:softmax",
        "num_class": num_classes,
        "max_depth": 5,
        "eta": 0.08,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "tree_method": "hist",
        "eval_metric": "mlogloss",
        "seed": RANDOM_STATE,
    }

    # -----------------------------
    # Train
    # -----------------------------
    model = xgb.train(
        params=params,
        dtrain=dtrain,
        num_boost_round=250,
        evals=[(dtrain, "train")],
        verbose_eval=False
    )

    # -----------------------------
    # Evaluation
    # -----------------------------
    t0 = time.perf_counter()
    y_pred = model.predict(dtest)
    t1 = time.perf_counter()

    inf_ms = (t1 - t0) / len(y_pred) * 1000

    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average="macro", zero_division=0)
    rec  = recall_score(y_test, y_pred, average="macro")
    f1   = f1_score(y_test, y_pred, average="macro")

    results.append([acc, prec, rec, f1, inf_ms])

    print(f"  Acc={acc:.3f} | F1={f1:.3f} | Inf={inf_ms:.3f} ms")

# -------------------------------------------------
# Aggregate results
# -------------------------------------------------
results = np.array(results)

print("\n📊 LOSO XGBoost Results (50% Calibration)")
print(f"Accuracy  : {results[:,0].mean():.3f} ± {results[:,0].std():.3f}")
print(f"Precision : {results[:,1].mean():.3f} ± {results[:,1].std():.3f}")
print(f"Recall    : {results[:,2].mean():.3f} ± {results[:,2].std():.3f}")
print(f"F1-score  : {results[:,3].mean():.3f} ± {results[:,3].std():.3f}")
print(f"Inference : {results[:,4].mean():.3f} ± {results[:,4].std():.3f} ms")


Running LOSO XGBoost with 50% calibration

▶ Subject p1
  Acc=0.316 | F1=0.180 | Inf=0.348 ms
▶ Subject p10
  Acc=0.708 | F1=0.376 | Inf=0.548 ms
▶ Subject p11
  Acc=0.450 | F1=0.144 | Inf=0.413 ms
▶ Subject p2
  Acc=0.330 | F1=0.172 | Inf=0.622 ms
▶ Subject p3
  Acc=0.647 | F1=0.217 | Inf=0.827 ms
▶ Subject p4
  Acc=0.364 | F1=0.218 | Inf=0.150 ms
▶ Subject p5
  Acc=0.539 | F1=0.404 | Inf=0.138 ms
▶ Subject p6
  Acc=0.426 | F1=0.125 | Inf=0.268 ms
▶ Subject p7
  Acc=0.276 | F1=0.180 | Inf=0.144 ms
▶ Subject p8
  Acc=0.321 | F1=0.355 | Inf=0.194 ms
▶ Subject p9
  Acc=0.220 | F1=0.163 | Inf=0.153 ms

📊 LOSO XGBoost Results (50% Calibration)
Accuracy  : 0.418 ± 0.149
Precision : 0.289 ± 0.106
Recall    : 0.222 ± 0.096
F1-score  : 0.230 ± 0.095
Inference : 0.346 ± 0.222 ms


In [11]:
# ==========================================================
# 🔁 LOSO — Population → 50% Calibration → Test (EEGNet)
# ==========================================================

import numpy as np, time, warnings
warnings.filterwarnings("ignore")

import tensorflow as tf
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.constraints import max_norm
from tensorflow.keras.regularizers import l2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Conv2D, DepthwiseConv2D, SeparableConv2D,
                                     BatchNormalization, Activation, AveragePooling2D,
                                     Dropout, Flatten, Dense)

# -------------------------------------------------
# CONFIG
# -------------------------------------------------
CALIBRATION_RATIO = 0.50
RANDOM_STATE = 42
EPOCHS = 150
BATCH_SIZE = 32

# -------------------------------------------------
# EEGNet (UNCHANGED)
# -------------------------------------------------
def EEGNet(nb_classes, Chans, Samples,
           dropoutRate=0.6, kernLength=64, F1=8, D=2,
           norm_rate=0.25, l2_reg=1e-4):

    F2 = F1 * D
    reg = l2(l2_reg)
    inp = Input(shape=(Chans, Samples, 1))

    x = Conv2D(F1, (1, kernLength), padding='same', use_bias=False,
               kernel_regularizer=reg)(inp)
    x = BatchNormalization()(x)

    x = DepthwiseConv2D((Chans, 1), use_bias=False, depth_multiplier=D,
                        depthwise_constraint=max_norm(1.),
                        depthwise_regularizer=reg)(x)
    x = BatchNormalization()(x)
    x = Activation('elu')(x)
    x = AveragePooling2D((1, 4))(x)
    x = Dropout(dropoutRate)(x)

    x = SeparableConv2D(F2, (1, 16), padding='same', use_bias=False,
                        depthwise_regularizer=reg,
                        pointwise_regularizer=reg)(x)
    x = BatchNormalization()(x)
    x = Activation('elu')(x)
    x = AveragePooling2D((1, 8))(x)
    x = Dropout(dropoutRate)(x)

    x = Flatten()(x)
    x = Dense(nb_classes,
              kernel_constraint=max_norm(norm_rate),
              kernel_regularizer=reg)(x)
    out = Activation('softmax')(x)

    return Model(inp, out)

# -------------------------------------------------
# Helper: split
# -------------------------------------------------
def split_population_calibration_test(
    X_all, y_all, patient_ids, target_subject,
    calibration_ratio=0.5, random_state=42
):
    X_all = np.asarray(X_all, dtype=np.float32)
    y_all = np.asarray(y_all)
    patient_ids = np.asarray(patient_ids)

    mask_target = patient_ids == target_subject

    X_pop = X_all[~mask_target]
    y_pop = y_all[~mask_target]

    X_target = X_all[mask_target]
    y_target = y_all[mask_target]

    X_cal, X_test, y_cal, y_test = train_test_split(
        X_target, y_target,
        test_size=1 - calibration_ratio,
        stratify=y_target,
        random_state=random_state
    )

    return X_pop, y_pop, X_cal, y_cal, X_test, y_test

# -------------------------------------------------
# Loop over all patients
# -------------------------------------------------
unique_patients = np.unique(patient_ids)
results = []

print(f"Running LOSO EEGNet with {int(CALIBRATION_RATIO*100)}% calibration\n")

for pid in unique_patients:
    print(f"▶ Subject {pid}")

    # -----------------------------
    # Split
    # -----------------------------
    X_pop, y_pop, X_cal, y_cal, X_test, y_test = split_population_calibration_test(
        X_all, y_all, patient_ids,
        pid,
        calibration_ratio=CALIBRATION_RATIO,
        random_state=RANDOM_STATE
    )

    # -----------------------------
    # Ensure (N, C, T, 1)
    # -----------------------------
    if X_pop.shape[1] != 14:
        X_pop  = X_pop.transpose(0, 2, 1)
        X_cal  = X_cal.transpose(0, 2, 1)
        X_test = X_test.transpose(0, 2, 1)

    X_pop  = np.expand_dims(X_pop,  -1)
    X_cal  = np.expand_dims(X_cal,  -1)
    X_test = np.expand_dims(X_test, -1)

    # -----------------------------
    # Encode labels (GLOBAL)
    # -----------------------------
    le = LabelEncoder()
    y_pop  = le.fit_transform(y_pop)
    y_cal  = le.transform(y_cal)
    y_test = le.transform(y_test)

    num_classes = len(le.classes_)

    y_pop_cat  = to_categorical(y_pop,  num_classes)
    y_cal_cat  = to_categorical(y_cal,  num_classes)
    y_test_cat = to_categorical(y_test, num_classes)

    # -----------------------------
    # Class weights (population + calibration)
    # -----------------------------
    y_train_all = np.concatenate([y_pop, y_cal])
    cw_vals = compute_class_weight(
        class_weight="balanced",
        classes=np.unique(y_train_all),
        y=y_train_all
    )
    class_weights = dict(enumerate(cw_vals))

    # -----------------------------
    # Build model
    # -----------------------------
    Chans, Samples = X_pop.shape[1], X_pop.shape[2]
    model = EEGNet(num_classes, Chans, Samples)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=5e-4),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=20,
        restore_best_weights=True
    )

    # -----------------------------
    # Train (population + calibration)
    # -----------------------------
    model.fit(
        np.vstack([X_pop, X_cal]),
        np.vstack([y_pop_cat, y_cal_cat]),
        validation_split=0.1,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        class_weight=class_weights,
        callbacks=[early_stop],
        verbose=0
    )

    # -----------------------------
    # Evaluation
    # -----------------------------
    t0 = time.perf_counter()
    y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
    t1 = time.perf_counter()

    inf_ms = (t1 - t0) / len(y_pred) * 1000

    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average="macro", zero_division=0)
    rec  = recall_score(y_test, y_pred, average="macro")
    f1   = f1_score(y_test, y_pred, average="macro")

    results.append([acc, prec, rec, f1, inf_ms])

    print(f"  Acc={acc:.3f} | F1={f1:.3f} | Inf={inf_ms:.3f} ms")

    tf.keras.backend.clear_session()

# -------------------------------------------------
# Aggregate results
# -------------------------------------------------
results = np.array(results)

print("\n📊 LOSO EEGNet Results (50% Calibration)")
print(f"Accuracy  : {results[:,0].mean():.3f} ± {results[:,0].std():.3f}")
print(f"Precision : {results[:,1].mean():.3f} ± {results[:,1].std():.3f}")
print(f"Recall    : {results[:,2].mean():.3f} ± {results[:,2].std():.3f}")
print(f"F1-score  : {results[:,3].mean():.3f} ± {results[:,3].std():.3f}")
print(f"Inference : {results[:,4].mean():.3f} ± {results[:,4].std():.3f} ms")


Running LOSO EEGNet with 50% calibration

▶ Subject p1
  Acc=0.214 | F1=0.094 | Inf=2.481 ms

▶ Subject p10
  Acc=0.153 | F1=0.105 | Inf=3.125 ms
▶ Subject p11
  Acc=0.525 | F1=0.162 | Inf=3.187 ms
▶ Subject p2
  Acc=0.193 | F1=0.104 | Inf=2.912 ms
▶ Subject p3
  Acc=0.431 | F1=0.204 | Inf=4.555 ms
▶ Subject p4
  Acc=0.318 | F1=0.147 | Inf=2.793 ms
▶ Subject p5
  Acc=0.202 | F1=0.121 | Inf=2.881 ms
▶ Subject p6
  Acc=0.340 | F1=0.142 | Inf=5.024 ms
▶ Subject p7
  Acc=0.207 | F1=0.058 | Inf=2.719 ms
▶ Subject p8
  Acc=0.013 | F1=0.015 | Inf=3.099 ms
▶ Subject p9
  Acc=0.159 | F1=0.056 | Inf=2.824 ms

📊 LOSO EEGNet Results (50% Calibration)
Accuracy  : 0.251 ± 0.136
Precision : 0.136 ± 0.065
Recall    : 0.133 ± 0.067
F1-score  : 0.110 ± 0.051
Inference : 3.236 ± 0.763 ms


In [12]:
# ==========================================================
# 🔁 LOSO — Population → 50% Calibration → Test (EEG-TCNet)
# ==========================================================

import numpy as np, time, warnings
warnings.filterwarnings("ignore")

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Conv2D, DepthwiseConv2D, SeparableConv2D,
                                     BatchNormalization, Activation, AveragePooling2D,
                                     Dropout, Flatten, Dense, Add)
from tensorflow.keras.constraints import max_norm
from tensorflow.keras.regularizers import l2
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# -------------------------------------------------
# CONFIG
# -------------------------------------------------
CALIBRATION_RATIO = 0.50
RANDOM_STATE = 42
EPOCHS = 150
BATCH_SIZE = 32

# -------------------------------------------------
# EEG-TCNet (UNCHANGED)
# -------------------------------------------------
def EEGTCNet(nb_classes, Chans, Samples,
             n_layers=2, F1=8, D=2,
             kernel_s=4, kernel_t=32,
             dropout=0.5, dropout_eeg=0.3,
             activation='elu', l2_reg=1e-4):

    F2 = F1 * D
    reg = l2(l2_reg)
    inp = Input(shape=(Chans, Samples, 1))

    x = Conv2D(F1, (1, kernel_t), padding='same',
               use_bias=False, kernel_regularizer=reg)(inp)
    x = BatchNormalization()(x)

    x = DepthwiseConv2D((Chans, 1), depth_multiplier=D,
                        use_bias=False,
                        depthwise_constraint=max_norm(1.),
                        depthwise_regularizer=reg)(x)
    x = BatchNormalization()(x)
    x = Activation(activation)(x)
    x = AveragePooling2D((1, 4))(x)
    x = Dropout(dropout_eeg)(x)

    for _ in range(n_layers):
        res = x

        x = SeparableConv2D(F2, (1, kernel_s), padding='same',
                            use_bias=False,
                            depthwise_regularizer=reg,
                            pointwise_regularizer=reg)(x)
        x = BatchNormalization()(x)
        x = Activation(activation)(x)

        x = SeparableConv2D(F2, (1, kernel_s), padding='same',
                            use_bias=False,
                            depthwise_regularizer=reg,
                            pointwise_regularizer=reg)(x)
        x = BatchNormalization()(x)

        x = Add()([x, res])
        x = Activation(activation)(x)
        x = AveragePooling2D((1, 2))(x)
        x = Dropout(dropout)(x)

    x = Flatten()(x)
    x = Dense(nb_classes, kernel_constraint=max_norm(0.25))(x)
    out = Activation('softmax')(x)

    return Model(inp, out)

# -------------------------------------------------
# Helper: split
# -------------------------------------------------
def split_population_calibration_test(
    X_all, y_all, patient_ids, target_subject,
    calibration_ratio=0.5, random_state=42
):
    X_all = np.asarray(X_all, dtype=np.float32)
    y_all = np.asarray(y_all)
    patient_ids = np.asarray(patient_ids)

    mask_target = patient_ids == target_subject

    X_pop = X_all[~mask_target]
    y_pop = y_all[~mask_target]

    X_target = X_all[mask_target]
    y_target = y_all[mask_target]

    X_cal, X_test, y_cal, y_test = train_test_split(
        X_target, y_target,
        test_size=1 - calibration_ratio,
        stratify=y_target,
        random_state=random_state
    )

    return X_pop, y_pop, X_cal, y_cal, X_test, y_test

# -------------------------------------------------
# Loop over all patients
# -------------------------------------------------
unique_patients = np.unique(patient_ids)
results = []

print(f"Running LOSO EEG-TCNet with {int(CALIBRATION_RATIO*100)}% calibration\n")

for pid in unique_patients:
    print(f"▶ Subject {pid}")

    # -----------------------------
    # Split
    # -----------------------------
    X_pop, y_pop, X_cal, y_cal, X_test, y_test = split_population_calibration_test(
        X_all, y_all, patient_ids,
        pid,
        calibration_ratio=CALIBRATION_RATIO,
        random_state=RANDOM_STATE
    )

    # -----------------------------
    # Ensure (N, C, T, 1)
    # -----------------------------
    if X_pop.shape[1] != 14:
        X_pop  = X_pop.transpose(0, 2, 1)
        X_cal  = X_cal.transpose(0, 2, 1)
        X_test = X_test.transpose(0, 2, 1)

    X_pop  = np.expand_dims(X_pop,  -1)
    X_cal  = np.expand_dims(X_cal,  -1)
    X_test = np.expand_dims(X_test, -1)

    # -----------------------------
    # Encode labels
    # -----------------------------
    le = LabelEncoder()
    y_pop  = le.fit_transform(y_pop)
    y_cal  = le.transform(y_cal)
    y_test = le.transform(y_test)

    num_classes = len(le.classes_)

    y_pop_cat  = to_categorical(y_pop,  num_classes)
    y_cal_cat  = to_categorical(y_cal,  num_classes)
    y_test_cat = to_categorical(y_test, num_classes)

    # -----------------------------
    # Class weights
    # -----------------------------
    y_train_all = np.concatenate([y_pop, y_cal])
    cw_vals = compute_class_weight(
        class_weight='balanced',
        classes=np.unique(y_train_all),
        y=y_train_all
    )
    class_weights = dict(enumerate(cw_vals))

    # -----------------------------
    # Build model
    # -----------------------------
    Chans, Samples = X_pop.shape[1], X_pop.shape[2]
    model = EEGTCNet(num_classes, Chans, Samples)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    early_stop = EarlyStopping(
        monitor='val_accuracy',
        patience=50,
        restore_best_weights=True
    )

    # -----------------------------
    # Train (population + calibration)
    # -----------------------------
    model.fit(
        np.vstack([X_pop, X_cal]),
        np.vstack([y_pop_cat, y_cal_cat]),
        validation_split=0.1,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        class_weight=class_weights,
        callbacks=[early_stop],
        verbose=0
    )

    # -----------------------------
    # Evaluation
    # -----------------------------
    t0 = time.perf_counter()
    y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
    t1 = time.perf_counter()

    inf_ms = (t1 - t0) / len(y_pred) * 1000

    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='macro', zero_division=0)
    rec  = recall_score(y_test, y_pred, average='macro')
    f1   = f1_score(y_test, y_pred, average='macro')

    results.append([acc, prec, rec, f1, inf_ms])

    print(f"  Acc={acc:.3f} | F1={f1:.3f} | Inf={inf_ms:.3f} ms")

    tf.keras.backend.clear_session()

# -------------------------------------------------
# Aggregate results
# -------------------------------------------------
results = np.array(results)

print("\n📊 LOSO EEG-TCNet Results (50% Calibration)")
print(f"Accuracy  : {results[:,0].mean():.3f} ± {results[:,0].std():.3f}")
print(f"Precision : {results[:,1].mean():.3f} ± {results[:,1].std():.3f}")
print(f"Recall    : {results[:,2].mean():.3f} ± {results[:,2].std():.3f}")
print(f"F1-score  : {results[:,3].mean():.3f} ± {results[:,3].std():.3f}")
print(f"Inference : {results[:,4].mean():.3f} ± {results[:,4].std():.3f} ms")

Running LOSO EEG-TCNet with 50% calibration

▶ Subject p1
  Acc=0.316 | F1=0.194 | Inf=3.572 ms
▶ Subject p10
  Acc=0.306 | F1=0.158 | Inf=4.668 ms
▶ Subject p11
  Acc=0.463 | F1=0.211 | Inf=4.649 ms
▶ Subject p2
  Acc=0.250 | F1=0.133 | Inf=3.918 ms
▶ Subject p3
  Acc=0.627 | F1=0.560 | Inf=6.739 ms
▶ Subject p4
  Acc=0.000 | F1=0.000 | Inf=4.284 ms
▶ Subject p5
  Acc=0.000 | F1=0.000 | Inf=3.713 ms
▶ Subject p6
  Acc=0.191 | F1=0.120 | Inf=8.223 ms
▶ Subject p7
  Acc=0.241 | F1=0.088 | Inf=4.006 ms
▶ Subject p8
  Acc=0.205 | F1=0.085 | Inf=4.698 ms
▶ Subject p9
  Acc=0.317 | F1=0.141 | Inf=5.771 ms

📊 LOSO EEG-TCNet Results (50% Calibration)
Accuracy  : 0.265 ± 0.173
Precision : 0.154 ± 0.189
Recall    : 0.219 ± 0.161
F1-score  : 0.154 ± 0.144
Inference : 4.931 ± 1.370 ms


In [13]:
# ==========================================================
# 🔁 LOSO — Population → 50% Calibration → Test
#    Filter-Bank Riemann + Tangent Space + Logistic Regression
# ==========================================================

import numpy as np, time, warnings
warnings.filterwarnings("ignore")

from pyriemann.estimation import Covariances
from pyriemann.tangentspace import TangentSpace
from scipy.signal import butter, filtfilt

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, GridSearchCV, train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# -------------------------------------------------
# CONFIG
# -------------------------------------------------
CALIBRATION_RATIO = 0.50
RANDOM_STATE = 42
FS = 256.0

bands = [
    (8, 12),   # μ
    (13, 20),  # low-β
    (20, 30),  # high-β
    (4, 7),    # θ
]

# -------------------------------------------------
# Helpers
# -------------------------------------------------
def ensure_nct(X):
    n, a, b = X.shape
    if a <= 64 and b >= 50: return X
    if b <= 64 and a >= 50: return X.transpose(0, 2, 1)
    return X

def bandpass(data, lo, hi, fs=FS, order=4):
    b, a = butter(order, [lo/(fs/2), hi/(fs/2)], btype="band")
    return filtfilt(b, a, data, axis=2)

def fb_riemann_ts(Xtr, Xte, ytr, bands):
    feats_tr, feats_te = [], []
    for lo, hi in bands:
        Xtr_f = bandpass(Xtr, lo, hi)
        Xte_f = bandpass(Xte, lo, hi)

        cov_tr = Covariances(estimator="oas").fit_transform(Xtr_f)
        cov_te = Covariances(estimator="oas").transform(Xte_f)

        ts = TangentSpace()
        ts.fit(cov_tr, ytr)

        feats_tr.append(ts.transform(cov_tr))
        feats_te.append(ts.transform(cov_te))

    return np.concatenate(feats_tr, axis=1), np.concatenate(feats_te, axis=1)

def split_population_calibration_test(X_all, y_all, patient_ids, target_subject,
                                      calibration_ratio=0.5, random_state=42):
    X_all = np.asarray(X_all, dtype=np.float32)
    y_all = np.asarray(y_all)
    patient_ids = np.asarray(patient_ids)

    mask_target = patient_ids == target_subject

    X_pop = X_all[~mask_target]
    y_pop = y_all[~mask_target]

    X_target = X_all[mask_target]
    y_target = y_all[mask_target]

    X_cal, X_test, y_cal, y_test = train_test_split(
        X_target, y_target,
        test_size=1 - calibration_ratio,
        stratify=y_target,
        random_state=random_state
    )

    return X_pop, y_pop, X_cal, y_cal, X_test, y_test

# -------------------------------------------------
# Loop over all patients
# -------------------------------------------------
unique_patients = np.unique(np.asarray(patient_ids))
results = []

print(f"Running LOSO FB-Riemann+TS+LogReg with {int(CALIBRATION_RATIO*100)}% calibration\n")

for pid in unique_patients:
    print(f"▶ Subject {pid}")

    # -----------------------------
    # Split
    # -----------------------------
    X_pop, y_pop, X_cal, y_cal, X_test, y_test = split_population_calibration_test(
        X_all, y_all, patient_ids,
        pid,
        calibration_ratio=CALIBRATION_RATIO,
        random_state=RANDOM_STATE
    )

    # -----------------------------
    # Encode labels (GLOBAL per run)
    # -----------------------------
    le = LabelEncoder()
    y_pop  = le.fit_transform(y_pop)
    y_cal  = le.transform(y_cal)
    y_test = le.transform(y_test)

    # -----------------------------
    # Shape + de-mean
    # -----------------------------
    X_pop  = ensure_nct(np.asarray(X_pop, dtype=np.float32))
    X_cal  = ensure_nct(np.asarray(X_cal, dtype=np.float32))
    X_test = ensure_nct(np.asarray(X_test, dtype=np.float32))

    X_pop  -= X_pop.mean(axis=2, keepdims=True)
    X_cal  -= X_cal.mean(axis=2, keepdims=True)
    X_test -= X_test.mean(axis=2, keepdims=True)

    # -----------------------------
    # Feature extraction (FB-Riemann TS)
    # Fit TS using POPULATION labels (same as SVM cell)
    # -----------------------------
    Xpop_ts, Xcal_ts = fb_riemann_ts(X_pop, X_cal, y_pop, bands)
    _, Xtest_ts      = fb_riemann_ts(X_pop, X_test, y_pop, bands)

    # -----------------------------
    # Scaling (population + calibration)
    # -----------------------------
    scaler = StandardScaler()
    scaler.fit(np.vstack([Xpop_ts, Xcal_ts]))

    Xpop_ts  = scaler.transform(Xpop_ts)
    Xcal_ts  = scaler.transform(Xcal_ts)
    Xtest_ts = scaler.transform(Xtest_ts)

    # -----------------------------
    # Train LogReg (population + calibration)
    # -----------------------------
    X_train = np.vstack([Xpop_ts, Xcal_ts])
    y_train = np.concatenate([y_pop, y_cal])

    base_clf = LogisticRegression(
        solver="lbfgs",
        max_iter=3000,
        class_weight="balanced",
        n_jobs=1,
        multi_class="auto",
        random_state=RANDOM_STATE
    )

    param_grid = {"C": np.logspace(-2, 2, 7)}
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

    clf = GridSearchCV(
        base_clf,
        param_grid=param_grid,
        cv=cv,
        n_jobs=1,
        refit=True
    )

    clf.fit(X_train, y_train)

    # -----------------------------
    # Evaluation
    # -----------------------------
    t0 = time.perf_counter()
    y_pred = clf.predict(Xtest_ts)
    t1 = time.perf_counter()

    inf_ms = (t1 - t0) / len(y_pred) * 1000

    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average="macro", zero_division=0)
    rec  = recall_score(y_test, y_pred, average="macro")
    f1   = f1_score(y_test, y_pred, average="macro")

    results.append([acc, prec, rec, f1, inf_ms])

    print(f"  Acc={acc:.3f} | F1={f1:.3f} | Inf={inf_ms:.3f} ms")

# -------------------------------------------------
# Aggregate results
# -------------------------------------------------
results = np.array(results)

print("\n📊 LOSO FB-Riemann + TS + LogReg Results (50% Calibration)")
print(f"Accuracy  : {results[:,0].mean():.3f} ± {results[:,0].std():.3f}")
print(f"Precision : {results[:,1].mean():.3f} ± {results[:,1].std():.3f}")
print(f"Recall    : {results[:,2].mean():.3f} ± {results[:,2].std():.3f}")
print(f"F1-score  : {results[:,3].mean():.3f} ± {results[:,3].std():.3f}")
print(f"Inference : {results[:,4].mean():.3f} ± {results[:,4].std():.3f} ms")


Running LOSO FB-Riemann+TS+LogReg with 50% calibration

▶ Subject p1
  Acc=0.806 | F1=0.807 | Inf=0.005 ms
▶ Subject p10
  Acc=0.847 | F1=0.438 | Inf=0.005 ms
▶ Subject p11
  Acc=0.812 | F1=0.480 | Inf=0.005 ms
▶ Subject p2
  Acc=0.864 | F1=0.693 | Inf=0.005 ms
▶ Subject p3
  Acc=0.804 | F1=0.552 | Inf=0.008 ms
▶ Subject p4
  Acc=0.818 | F1=0.810 | Inf=0.005 ms
▶ Subject p5
  Acc=0.843 | F1=0.681 | Inf=0.005 ms
▶ Subject p6
  Acc=0.787 | F1=0.536 | Inf=0.009 ms
▶ Subject p7
  Acc=0.736 | F1=0.590 | Inf=0.005 ms
▶ Subject p8
  Acc=0.744 | F1=0.599 | Inf=0.005 ms
▶ Subject p9
  Acc=0.537 | F1=0.351 | Inf=0.005 ms

📊 LOSO FB-Riemann + TS + LogReg Results (50% Calibration)
Accuracy  : 0.782 ± 0.086
Precision : 0.610 ± 0.133
Recall    : 0.587 ± 0.144
F1-score  : 0.594 ± 0.138
Inference : 0.006 ± 0.001 ms
